# 02 Radiomics + ComBat + subtype baselines (CPU, no GPU needed)
1. **Features** (`scripts/extract_radiomics.py`, `src/radiomics.py`) for every cached patient. They're
   computed on cache v2 (same 0.7 x 0.7 x 2 mm voxels for everyone), with intensities converted back to
   raw-proportional units:
   - **shape:** volume, sphericity, axes, number of parts;
   - **kinetics:** relative enhancement (early - pre)/pre and (late - pre)/pre; washout / plateau / persistent
     fractions;
   - **heterogeneity:** entropy and GLCM texture of the enhancement map;
   - **peritumoural 5 mm ring:** enhancement just outside the tumour.

   Tumour = the I-SPY mask; for Duke (boxes only), the box voxels enhancing >= 70% (the I-SPY rule).
2. **Baselines** (`src/baselines/radiomics.py`), fitted on train and scored on val:
   - `subtype`: HR/HER2 lookup;
   - `rad`: radiomics, ComBat-harmonised by protocol group (matrix x slice thickness x pixel size, per
     cohort);
   - `rad_sub`: `rad` + HR/HER2;
   - `rad_nocombat`, `rad_sub_nocombat`: the same without ComBat.

   Each model is compared with `subtype` by paired bootstrap.

Session: Accelerator **None**, Internet **ON**, secret `GITHUB_TOKEN`, **+ Add Input** `mnj-cache-v2`.
`LIMIT = 30` first (a minute), then `LIMIT = 0` and **Save & Run All (Commit)** (about 30-60 min).
The features CSV is pushed with the results, so A can rerun the baselines locally.

In [ ]:
# ==== PARAMETERS (edit these) ====
REPO = "AkulaReddy/MRI_chemo_response"
GIT_TAG = "d1.11"        # code version; A posts the tag to use
PERSON = ""             # set to "A", "B" or "C"
LIMIT = 0               # patients per cohort; 30 for a quick check
WORKERS = 4
CACHE_VERSION = 2
PUSH_RESULTS = True

In [ ]:
# ==== SETUP (do not edit) ====
import base64, json, os, shutil, subprocess, sys
assert PERSON in ("A", "B", "C"), 'set PERSON = "A", "B" or "C" in the parameters cell'
from kaggle_secrets import UserSecretsClient
_tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
_hdr = "Authorization: Basic " + base64.b64encode(f"x-access-token:{_tok}".encode()).decode()

def git(*args, cwd="/tmp/repo"):
    """git with the token as a one-off header (never stored in .git/config or the output)."""
    r = subprocess.run(["git", "-c", f"http.extraHeader={_hdr}", *args], cwd=cwd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).replace(_tok, "***")
    if r.returncode:
        raise RuntimeError(f"git {args[0]} failed:\n{out}")
    return out

shutil.rmtree("/tmp/repo", ignore_errors=True)
git("clone", "--quiet", "--depth", "1", "--branch", GIT_TAG, f"https://github.com/{REPO}.git", "/tmp/repo", cwd="/tmp")
os.chdir("/tmp/repo")
print("code at", GIT_TAG, git("log", "-1", "--format=%h %s"))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"], check=True)
import torch; print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
import time
CONFIG, SEED = "radiomics", "-"     # used in the push commit message
RES = f"/kaggle/working/results/{PERSON}/radiomics_{time.strftime('%m%d-%H%M')}" + ("_debug" if LIMIT else "")
os.makedirs(RES, exist_ok=True)
cmd = [sys.executable, "scripts/extract_radiomics.py", "--cache-root", "/kaggle/input", "--cache-version",
       str(CACHE_VERSION), "--out", f"{RES}/features.csv", "--workers", str(WORKERS)] + (["--limit", str(LIMIT)] if LIMIT else [])
print(" ".join(cmd)); subprocess.run(cmd, check=True)
cmd = [sys.executable, "-m", "src.baselines.radiomics", "--features", f"{RES}/features.csv", "--out", RES]
print(" ".join(cmd)); subprocess.run(cmd, check=True)

In [ ]:
import glob
import pandas as pd
from IPython.display import display
f = pd.read_csv(f"{RES}/features.csv")
print("mask source per cohort:"); display(f.groupby("cohort").mask_source.value_counts().unstack(fill_value=0))
from src.radiomics import FEATURES
print("feature medians per cohort (raw, before ComBat):"); display(f.groupby("cohort")[FEATURES].median().T.round(3))
rows = []
for m in sorted(glob.glob(f"{RES}/r_*/metrics.json")):
    j = json.load(open(m)); v = j["val"]
    rows.append({"model": j["config"], "val_auc": round(v["auc"], 3), "ci": f"{v['auc_ci'][0]:.3f}-{v['auc_ci'][1]:.3f}",
                 **{f"auc_{c}": round(x["auc"], 3) for c, x in v["by_cohort"].items()},
                 "batch_acc_before": j.get("batch_acc_before"), "batch_acc_after": j.get("batch_acc_after"),
                 "chance": j.get("batch_chance")})
display(pd.DataFrame(rows))

In [ ]:
# ==== PUSH the small result files to GitHub (results/<PERSON>/<run_id>/) ====
if PUSH_RESULTS and not LIMIT:
    git("config", "user.name", f"Person {PERSON} (Kaggle)")
    git("config", "user.email", "kaggle@users.noreply.github.com")
    # the clone holds only the tag, so fetch main into origin/main explicitly; on a rejected push
    # (someone else pushed first) rebuild the commit on the new main instead of rebasing a shallow clone
    for attempt in range(3):
        git("fetch", "--quiet", "--depth", "1", "origin", "+refs/heads/main:refs/remotes/origin/main")
        git("checkout", "--quiet", "-f", "-B", "main", "origin/main")
        shutil.copytree(f"/kaggle/working/results/{PERSON}", f"results/{PERSON}", dirs_exist_ok=True)
        git("add", f"results/{PERSON}")
        print(git("commit", "-m", f"results: {PERSON} {CONFIG} seed {SEED} ({GIT_TAG})"))
        try:
            git("push", "--quiet", "origin", "main"); print("pushed"); break
        except RuntimeError as e:
            print(f"push attempt {attempt + 1} failed:", e)
    else:
        print("NOT pushed: download results/ from the Output and send it to A")